# Build a Coding Agent from Scratch -- Exercise Notebook

> *"The Emperor Has No Clothes"* -- it turns out the core of every AI coding assistant
> (Claude Code, Cursor, Aider, etc.) is a surprisingly simple loop.

**This is the exercise version.** The key functions are provided as skeleton code
with `# TODO` comments. Your job is to implement them!

**How to use this notebook:**
1. Read the explanatory sections to understand the concepts
2. Implement the `# TODO` sections in each exercise cell
3. Run the test cell after each exercise to check your work
4. Once all exercises pass, run the demo section at the end to see your agent in action!

**What you will build:** A fully functional coding agent that can read files, list directories,
and edit code -- all powered by a free LLM via [OpenRouter](https://openrouter.ai).

**What you will need:** Basic Python knowledge, and about 60 minutes.

---

*This tutorial is based on [The Emperor Has No Clothes](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)
by [Mihail Eric](https://www.mihaileric.com/), which demonstrates how to replicate the core of Claude Code
in ~200 lines. We adapt the code to use [OpenRouter](https://openrouter.ai) free models so you can follow
along without spending a cent.*


## 1. What IS a Coding Agent?

A coding agent is a program that uses a Large Language Model (LLM) to **autonomously**
accomplish programming tasks. Unlike a simple chatbot that gives you text answers,
a coding agent can *act* on your codebase: reading files, navigating directories, writing code.

The key insight -- and the reason this tutorial exists -- is that the core architecture
is **shockingly simple**. Here is the entire flow:

```
+------------------------------------------------------+
|                  THE AGENTIC LOOP                    |
|                                                      |
|   +----------+     +----------+     +----------+     |
|   |          |     |          |     |          |     |
|   |  USER    |---->|   LLM    |---->|  TOOLS   |     |
|   |  INPUT   |     | (thinks) |     |(execute) |     |
|   |          |     |          |     |          |     |
|   +----------+     +-----+----+     +----+-----+     |
|                          |                |          |
|                          |    <-----------+          |
|                          |   tool results fed back   |
|                          |                           |
|                          v                           |
|                   +----------+                       |
|                   | RESPONSE |  (when no more tools  |
|                   | to user  |   are needed)         |
|                   +----------+                       |
+------------------------------------------------------+
```

**That's it.** A `while True` loop where:

1. The user sends a message
2. The LLM decides what to do (maybe call a tool, maybe respond directly)
3. If it requests a tool: execute it, feed the result back, and go to step 2
4. If no tool is needed: show the response to the user, go to step 1

This is a variant of the **ReAct (Reasoning + Acting)** pattern from
[Yao et al., 2022](https://arxiv.org/abs/2210.03629).

> **Key reference:** Anthropic's
> [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents)
> is the best industry write-up of these patterns. They recommend:
> *"find the simplest solution possible, and only increase complexity when needed."*


## 2. The Four Essential Tools

What tools does a coding agent need? Surprisingly few. Mihail Eric's
[original article](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/) uses just three
(read, list, edit). Mario Zechner's [pi coding agent](https://github.com/badlogic/pi-mono/tree/main/packages/coding-agent)
ships with four defaults (read, write, edit, **bash**). We will use four:

| Tool | What it does | Why it's needed |
|------|-------------|-----------------|
| `read_file` | Read a file's contents | The agent needs to *see* code |
| `list_files` | List files in a directory | The agent needs to *navigate* |
| `edit_file` | Create/modify a file | The agent needs to *write* code |
| `run_bash` | Execute a shell command | The agent can *do* anything a terminal can |

The `bash` tool is the great force multiplier: with it, the agent can run tests,
install packages, search with `grep`, check `git` status -- basically anything
a developer can do in a Unix terminal. Production agents like
[Claude Code](https://docs.anthropic.com/en/docs/claude-code),
[Aider](https://github.com/Aider-AI/aider), and
[SWE-agent](https://github.com/SWE-agent/SWE-agent) all include some form of shell access.

> **Critical insight:** The LLM never actually touches your filesystem or runs commands.
> It just *asks* for things to happen, and your Python code makes them happen.
> The LLM outputs text like `tool: read_file({"filename": "main.py"})`,
> and your code parses that, calls the real function, and feeds the result back.


## 3. Setup

Let's install our (minimal) dependencies and connect to [OpenRouter](https://openrouter.ai).

OpenRouter gives us access to many LLMs through a single, OpenAI-compatible API.
**Several powerful models are completely free** -- including coding specialists like
Mistral's Devstral and Google's Gemini Flash.

> **To get a free API key:** Go to [openrouter.ai/keys](https://openrouter.ai/keys),
> sign up, and create a key. Free models have a rate limit of ~50 requests/day (enough
> for this tutorial). If you add $10 in credits (even once), the limit jumps to 1,000/day.


In [1]:
# Install dependencies (using uv for speed; falls back to pip if uv is not available)
import subprocess, shutil
installer = "uv pip install" if shutil.which("uv") else "pip install"
subprocess.check_call(f"{installer} -q openai".split())

0

In [2]:
import inspect
import json
import subprocess as sp
from pathlib import Path
from typing import Any, Dict, List, Tuple
from getpass import getpass

from openai import OpenAI

In [3]:
# --- Configuration ---

# Enter your OpenRouter API key (free to create at openrouter.ai/keys)
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY") or getpass("Enter OpenRouter API key: ") #  getpass("Enter your OpenRouter API key: ")

# Which free model to use. Some excellent free options (as of early 2026):
#
# "qwen/qwen3-coder:free"                          -- 480B MoE coding specialist, 262K context
# "deepseek/deepseek-r1-0528:free"                  -- Strong reasoning model, 164K context
# "meta-llama/llama-3.3-70b-instruct:free"           -- Solid all-rounder, 128K context
# "nousresearch/hermes-3-llama-3.1-405b:free"        -- Largest free model (405B), 131K context
# "openai/gpt-oss-120b:free"                         -- OpenAI open-source model, 131K context
# "mistralai/mistral-small-3.1-24b-instruct:free"    -- Fast, 128K context
# "nvidia/nemotron-3-nano-30b-a3b:free"              -- Good for agentic workflows, 256K context
#
# Browse all free models:
# https://openrouter.ai/models?fmt=cards&input_modalities=text&max_price=0&order=most-popular

MODEL = "qwen/qwen3-coder-next"

# Initialize the OpenAI-compatible client pointed at OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://github.com/coding-agent-tutorial",
        "X-Title": "Coding Agent Tutorial",
    },
)

print(f"Client configured. Using model: {MODEL}")


Client configured. Using model: qwen/qwen3-coder-next


In [7]:
# Quick test: make sure the API works
test_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'Hello, Agent!' and nothing else."}],
    max_tokens=20,
)
print(f"API works! Response: {test_response.choices[0].message.content}")

API works! Response: Hello, Agent!


## 4. Building the Tools

Now let's implement our four tools. Each tool is a plain Python function that:
- Takes simple arguments (strings)
- Performs a real filesystem operation
- Returns a dictionary describing what happened

The LLM will learn about these tools through their **docstrings** and **type signatures**.

> **Anthropic's guide on [Writing Tools for Agents](https://www.anthropic.com/engineering/writing-tools-for-agents)**
> found that refining tool descriptions alone can dramatically improve agent performance.
> Good docstrings aren't just for humans -- they're instructions for the LLM.

### 4.1 Path Resolution (provided)

First, a utility to turn relative paths into absolute ones. This is provided for you
since it's infrastructure, not the main learning goal.


In [8]:
# We will use a workspace directory so the agent does not touch your real files.
# In Colab, we place it under /content/ so you can browse the agent's files
# in the built-in file browser (click the folder icon in the left sidebar).
import tempfile

def _make_workspace() -> Path:
    """Create a workspace directory visible in Colab's file browser."""
    try:
        import google.colab  # noqa: F401
        ws = Path("/content/agent_workspace")
        ws.mkdir(exist_ok=True)
        return ws
    except ImportError:
        return Path(tempfile.mkdtemp(prefix="agent_workspace_")).resolve()

WORKSPACE = _make_workspace()
print(f"Agent workspace: {WORKSPACE}")

def resolve_abs_path(path_str: str) -> Path:
    """
    Resolve a path string to an absolute path, sandboxed within WORKSPACE.
    Raises ValueError if the resolved path escapes the workspace.

    Examples:
        "file.py"       -> /tmp/agent_workspace_xyz/file.py
        "./src/main.py" -> /tmp/agent_workspace_xyz/src/main.py
    """
    path = Path(path_str).expanduser()
    if not path.is_absolute():
        path = (WORKSPACE / path).resolve()
    else:
        path = path.resolve()
    # Safety: ensure we never escape the workspace
    if not str(path).startswith(str(WORKSPACE)):
        raise ValueError(f"Path {path} is outside the workspace {WORKSPACE}")
    return path

# Test it
print(resolve_abs_path("hello.py"))
print(resolve_abs_path("src/utils.py"))

# This should raise an error (path escape attempt):
try:
    resolve_abs_path("/etc/passwd")
except ValueError as e:
    print(f"Blocked: {e}")

Agent workspace: /content/agent_workspace
/content/agent_workspace/hello.py
/content/agent_workspace/src/utils.py
Blocked: Path /etc/passwd is outside the workspace /content/agent_workspace


### Test Infrastructure

The following helper functions are used by the test cells throughout this notebook.
Run this cell before working on any exercises.


In [9]:
# --- Test Infrastructure ---
# Run this cell first! The test cells below depend on these helpers.

_test_results = []

def run_test(name, fn):
    """Run a single test function, catching and reporting errors."""
    try:
        fn()
        _test_results.append((name, "PASS"))
        print(f"  PASS: {name}")
    except Exception as e:
        _test_results.append((name, f"FAIL: {e}"))
        print(f"  FAIL: {name}: {e}")

def show_results():
    """Print a summary of test results and clear for next exercise."""
    passed = sum(1 for _, r in _test_results if r == "PASS")
    total = len(_test_results)
    print(f"\n{'='*40}")
    print(f"  {passed}/{total} tests passed")
    if passed == total:
        print("  All tests passed!")
    print(f"{'='*40}")
    _test_results.clear()

### Exercise 1: `read_file` -- Let the Agent See Code

Your first task: implement a tool that reads a file and returns its contents.

**Requirements:**
- Use `resolve_abs_path()` to convert the filename to an absolute path
- Open and read the file contents
- Return a dict with `"file_path"` (the absolute path as a string) and `"content"` (the file text)
- If the file doesn't exist, let the `FileNotFoundError` propagate (don't catch it)


In [10]:
def read_file_tool(filename: str) -> Dict[str, Any]:
    """
    Read the full content of a file.
    :param filename: The name or path of the file to read.
    :return: A dict with file_path and content keys.
    """


    # TODO: Implement this function
    # Hints:
    #   1. Use resolve_abs_path(filename) to get the full path
    #   2. Open the file and read its contents
    #   3. Return {"file_path": str(full_path), "content": content}
    full_path = resolve_abs_path(filename)
    with open(full_path, "r") as f:
        content = f.read()
    return {"file_path": str(full_path), "content": content}

**Tests for Exercise 1:** Run this cell to check your `read_file_tool` implementation.

In [11]:
print("Testing read_file_tool...")

# Setup: create a test file
_test_file = WORKSPACE / "_test_read.txt"
_test_file.write_text("hello world\nsecond line\n")

def test_returns_dict_with_correct_keys():
    result = read_file_tool("_test_read.txt")
    assert isinstance(result, dict), f"Expected dict, got {type(result)}"
    assert "file_path" in result, "Missing 'file_path' key"
    assert "content" in result, "Missing 'content' key"

def test_reads_file_content():
    result = read_file_tool("_test_read.txt")
    assert result["content"] == "hello world\nsecond line\n", \
        f"Wrong content: {result['content']!r}"

def test_uses_resolve_abs_path():
    result = read_file_tool("_test_read.txt")
    assert result["file_path"].startswith(str(WORKSPACE)), \
        f"Path {result['file_path']} should start with {WORKSPACE}"

def test_raises_on_nonexistent_file():
    try:
        read_file_tool("_no_such_file_xyz.txt")
        assert False, "Should have raised an exception"
    except FileNotFoundError:
        pass  # Expected
    except Exception as e:
        assert False, f"Expected FileNotFoundError, got {type(e).__name__}: {e}"

run_test("returns dict with correct keys", test_returns_dict_with_correct_keys)
run_test("reads file content correctly", test_reads_file_content)
run_test("uses resolve_abs_path (path in workspace)", test_uses_resolve_abs_path)
run_test("raises FileNotFoundError on missing file", test_raises_on_nonexistent_file)
show_results()

# Cleanup
_test_file.unlink(missing_ok=True)

Testing read_file_tool...
  PASS: returns dict with correct keys
  PASS: reads file content correctly
  PASS: uses resolve_abs_path (path in workspace)
  PASS: raises FileNotFoundError on missing file

  4/4 tests passed
  All tests passed!


### Exercise 2: `list_files` -- Let the Agent Navigate

Implement a tool that lists the files and directories at a given path.

**Requirements:**
- Use `resolve_abs_path()` to convert the path
- Iterate over the directory entries (hint: use `Path.iterdir()` with `sorted()`)
- For each entry, record its `"filename"` (just the name, not the full path) and
  `"type"` (`"file"` or `"dir"`)
- Return a dict with `"path"` (absolute path as string) and `"files"` (the list)


In [12]:
def list_files_tool(path: str) -> Dict[str, Any]:
    """
    List the files and directories at the given path.
    :param path: The path to a directory to list.
    :return: A dict with path and files keys. Each file entry
             has filename and type (file or dir).
    """
    # TODO: Implement this function
    # Hints:
    #   1. Use resolve_abs_path(path) to get the full path
    #   2. Use sorted(full_path.iterdir()) to get directory entries
    #   3. For each item, check item.is_file() to determine the type
    #   4. Return {"path": str(full_path), "files": all_files}

    full_path = resolve_abs_path(path)
    all_files = []
    for item in sorted(full_path.iterdir()):
        if item.is_file():
            file_type = "file"
        elif item.is_dir():
            file_type = "dir"
        else:
            continue # Skip other types like symlinks
        all_files.append({"filename": item.name, "type": file_type})
    return {"path": str(full_path), "files": all_files}

**Tests for Exercise 2:** Run this cell to check your `list_files_tool` implementation.

In [13]:
print("Testing list_files_tool...")

# Setup: create test files and a directory
_test_f1 = WORKSPACE / "_test_list_a.txt"
_test_f2 = WORKSPACE / "_test_list_b.py"
_test_dir = WORKSPACE / "_test_list_subdir"
_test_f1.write_text("a")
_test_f2.write_text("b")
_test_dir.mkdir(exist_ok=True)

def test_returns_dict_with_correct_keys():
    result = list_files_tool(".")
    assert isinstance(result, dict), f"Expected dict, got {type(result)}"
    assert "path" in result, "Missing 'path' key"
    assert "files" in result, "Missing 'files' key"
    assert isinstance(result["files"], list), "files should be a list"

def test_entries_have_filename_and_type():
    result = list_files_tool(".")
    for entry in result["files"]:
        assert "filename" in entry, f"Entry missing 'filename': {entry}"
        assert "type" in entry, f"Entry missing 'type': {entry}"

def test_distinguishes_files_and_dirs():
    result = list_files_tool(".")
    names_to_types = {e["filename"]: e["type"] for e in result["files"]}
    assert names_to_types.get("_test_list_a.txt") == "file", \
        "_test_list_a.txt should be type 'file'"
    assert names_to_types.get("_test_list_subdir") == "dir", \
        "_test_list_subdir should be type 'dir'"

def test_lists_newly_created_files():
    new_file = WORKSPACE / "_test_list_new.txt"
    new_file.write_text("new")
    result = list_files_tool(".")
    filenames = [e["filename"] for e in result["files"]]
    assert "_test_list_new.txt" in filenames, \
        f"Newly created file not found in listing: {filenames}"
    new_file.unlink()

run_test("returns dict with correct keys", test_returns_dict_with_correct_keys)
run_test("entries have filename and type", test_entries_have_filename_and_type)
run_test("distinguishes files and dirs", test_distinguishes_files_and_dirs)
run_test("lists newly created files", test_lists_newly_created_files)
show_results()

# Cleanup
_test_f1.unlink(missing_ok=True)
_test_f2.unlink(missing_ok=True)
import shutil as _shutil
_shutil.rmtree(_test_dir, ignore_errors=True)

Testing list_files_tool...
  PASS: returns dict with correct keys
  PASS: entries have filename and type
  PASS: distinguishes files and dirs
  PASS: lists newly created files

  4/4 tests passed
  All tests passed!


### Exercise 3: `edit_file` -- Let the Agent Write Code

This is the most powerful tool. It has two modes:

1. **Create mode** (`old_str` is empty `""`): Write `new_str` as the entire file content.
   Create parent directories if needed.
2. **Edit mode** (`old_str` is not empty): Find `old_str` in the file and replace the
   **first occurrence** with `new_str`.

**Requirements:**
- Use `resolve_abs_path()` to convert the path
- Create mode: create parent dirs (`mkdir(parents=True, exist_ok=True)`), write file,
  return `{"path": ..., "action": "created_file"}`
- Edit mode: read file, if `old_str` not found return `{"path": ..., "action": "old_str not found"}`,
  otherwise replace first occurrence and return `{"path": ..., "action": "edited"}`

> This "search and replace" approach is how many production agents work --
> it's more precise than rewriting entire files and less error-prone than
> line-number-based edits.


In [18]:
def edit_file_tool(path: str, old_str: str, new_str: str) -> Dict[str, Any]:
    """
    Edit a file by replacing old_str with new_str. If old_str is empty,
    create (or overwrite) the file with new_str as its full content.
    :param path: The path to the file to edit or create.
    :param old_str: The exact string to find and replace (empty = create file).
    :param new_str: The replacement string (or full file content if creating).
    :return: A dict with path and action keys.
    """
    # TODO: Implement this function
    # Hints:
    #   1. Use resolve_abs_path(path) to get the full path
    #   2. If old_str == "":
    #      - Create parent dirs: full_path.parent.mkdir(parents=True, exist_ok=True)
    #      - Write new_str to the file
    #      - Return {"path": str(full_path), "action": "created_file"}
    #   3. If old_str != "":
    #      - Read the file content
    #      - If old_str not in content, return {"path": ..., "action": "old_str not found"}
    #      - Replace first occurrence: content.replace(old_str, new_str, 1)
    #      - Write the modified content back
    #      - Return {"path": str(full_path), "action": "edited"}

    full_path = resolve_abs_path(path)
    if old_str == "":
        full_path.parent.mkdir(parents=True, exist_ok=True)
        full_path.write_text(new_str)
        return {"path": str(full_path), "action": "created_file"}
    else: # This handles the case where old_str is not empty
        with open(full_path, "r") as f:
            content = f.read()
        if old_str not in content:
            return {"path": str(full_path), "action": "old_str not found"}
        modified_content = content.replace(old_str, new_str, 1)
        full_path.write_text(modified_content)
        return {"path": str(full_path), "action": "edited"}

**Tests for Exercise 3:** Run this cell to check your `edit_file_tool` implementation.

In [34]:
print("Testing edit_file_tool...")

def test_create_mode():
    result = edit_file_tool("_test_edit_create.txt", "", "hello create")
    assert result["action"] == "created_file", f"Expected 'created_file', got {result['action']}"
    content = (WORKSPACE / "_test_edit_create.txt").read_text()
    assert content == "hello create", f"File content wrong: {content!r}"
    (WORKSPACE / "_test_edit_create.txt").unlink()

def test_edit_mode():
    (WORKSPACE / "_test_edit.txt").write_text("foo bar baz")
    result = edit_file_tool("_test_edit.txt", "bar", "REPLACED")
    assert result["action"] == "edited", f"Expected 'edited', got {result['action']}"
    content = (WORKSPACE / "_test_edit.txt").read_text()
    assert content == "foo REPLACED baz", f"File content wrong: {content!r}"
    (WORKSPACE / "_test_edit.txt").unlink()

def test_old_str_not_found():
    (WORKSPACE / "_test_edit_nf.txt").write_text("hello world")
    result = edit_file_tool("_test_edit_nf.txt", "NONEXISTENT", "replacement")
    assert result["action"] == "old_str not found", f"Expected 'old_str not found', got {result['action']}"
    (WORKSPACE / "_test_edit_nf.txt").unlink()

def test_creates_parent_directories():
    result = edit_file_tool("deep/nested/file.py", "", "print('hi')")
    assert result["action"] == "created_file"
    content = (WORKSPACE / "deep" / "nested" / "file.py").read_text()
    assert content == "print('hi')", f"File content wrong: {content!r}"
    import shutil as _shutil
    _shutil.rmtree(WORKSPACE / "deep", ignore_errors=True)

def test_replaces_only_first_occurrence():
    (WORKSPACE / "_test_edit_first.txt").write_text("aaa bbb aaa bbb")
    edit_file_tool("_test_edit_first.txt", "aaa", "XXX")
    content = (WORKSPACE / "_test_edit_first.txt").read_text()
    assert content == "XXX bbb aaa bbb", \
        f"Should only replace first occurrence, got: {content!r}"
    (WORKSPACE / "_test_edit_first.txt").unlink()

run_test("create mode (empty old_str)", test_create_mode)
run_test("edit mode (search and replace)", test_edit_mode)
run_test("old_str not found", test_old_str_not_found)
run_test("creates parent directories", test_creates_parent_directories)
run_test("replaces only first occurrence", test_replaces_only_first_occurrence)
show_results()

Testing edit_file_tool...
  PASS: create mode (empty old_str)
  PASS: edit mode (search and replace)
  PASS: old_str not found
  PASS: creates parent directories
  PASS: replaces only first occurrence

  5/5 tests passed
  All tests passed!


### Exercise 4: `run_bash` -- Let the Agent Execute Commands

This is the great force multiplier. With shell access, the agent can run tests,
install packages, use `grep`, check `git log` -- basically anything you can do
in a terminal.

**Requirements:**
- Use `subprocess.run()` with `shell=True`, `cwd=str(WORKSPACE)`, `capture_output=True`,
  `text=True`, and `timeout=timeout`
- Return a dict with `"stdout"`, `"stderr"`, and `"return_code"`
- Truncate stdout to last 5000 chars if longer, stderr to last 2000 chars if longer
- Catch `subprocess.TimeoutExpired` and return
  `{"stdout": "", "stderr": "Command timed out after {timeout}s", "return_code": -1}`

> **Safety note:** We sandbox this to our workspace directory and add a timeout.
> In production, you would run commands inside a container (Docker, etc.)
> and require user confirmation for dangerous operations.


In [35]:
def run_bash_tool(command: str, timeout: int = 30) -> Dict[str, Any]:
    """
    Execute a shell command in the workspace directory and return its output.
    :param command: The shell command to execute (e.g. "ls -la", "python test.py").
    :param timeout: Maximum seconds to wait (default 30).
    :return: A dict with stdout, stderr, and return_code keys.
    """
    # TODO: Implement this function
    # Hints:
    #   1. Use sp.run(command, shell=True, cwd=str(WORKSPACE),
    #                 capture_output=True, text=True, timeout=timeout)
    #   2. Truncate: if len(result.stdout) > 5000, keep only the last 5000 chars
    #   3. Truncate: if len(result.stderr) > 2000, keep only the last 2000 chars
    #   4. Return {"stdout": ..., "stderr": ..., "return_code": result.returncode}
    #   5. Catch sp.TimeoutExpired and return appropriate error dict

    try:
        result = sp.run(command, shell=True, cwd=str(WORKSPACE),
                         capture_output=True, text=True, timeout=timeout)

        stdout_output = result.stdout
        stderr_output = result.stderr

        if len(stdout_output) > 5000:
            stdout_output = stdout_output[-5000:]

        if len(stderr_output) > 2000:
            stderr_output = stderr_output[-2000:]

        return {"stdout": stdout_output, "stderr": stderr_output, "return_code": result.returncode}

    except sp.TimeoutExpired:
        return {"stdout": "", "stderr": f"Command timed out after {timeout}s", "return_code": -1}

**Tests for Exercise 4:** Run this cell to check your `run_bash_tool` implementation.

In [36]:
print("Testing run_bash_tool...")

def test_runs_command_and_returns_output():
    result = run_bash_tool("echo 'hello bash'")
    assert "stdout" in result, "Missing 'stdout' key"
    assert "stderr" in result, "Missing 'stderr' key"
    assert "return_code" in result, "Missing 'return_code' key"
    assert "hello bash" in result["stdout"], f"Expected 'hello bash' in stdout: {result['stdout']!r}"
    assert result["return_code"] == 0, f"Expected return_code 0, got {result['return_code']}"

def test_runs_in_workspace_directory():
    result = run_bash_tool("pwd")
    assert str(WORKSPACE) in result["stdout"], \
        f"pwd should return workspace path. Got: {result['stdout']!r}"

def test_handles_failing_commands():
    result = run_bash_tool("ls /nonexistent_path_xyz_12345")
    assert result["return_code"] != 0, "Failing command should have non-zero return_code"

def test_handles_timeout():
    result = run_bash_tool("sleep 10", timeout=1)
    assert result["return_code"] == -1, f"Expected return_code -1 for timeout, got {result['return_code']}"
    assert "timed out" in result["stderr"].lower(), \
        f"Expected 'timed out' in stderr: {result['stderr']!r}"

def test_truncates_long_output():
    # Generate stdout > 5000 chars
    result = run_bash_tool("python3 -c \"print('A' * 10000)\"")
    assert len(result["stdout"]) <= 5001, \
        f"stdout should be truncated to ~5000 chars, got {len(result['stdout'])}"

run_test("runs command and returns output", test_runs_command_and_returns_output)
run_test("runs in workspace directory", test_runs_in_workspace_directory)
run_test("handles failing commands", test_handles_failing_commands)
run_test("handles timeout", test_handles_timeout)
run_test("truncates long output", test_truncates_long_output)
show_results()

Testing run_bash_tool...
  PASS: runs command and returns output
  PASS: runs in workspace directory
  PASS: handles failing commands
  PASS: handles timeout
  PASS: truncates long output

  5/5 tests passed
  All tests passed!


## 5. The Tool Registry

Now we register our tools in a dictionary. This is important for two reasons:

1. **Dispatch** -- when the LLM says `tool: read_file(...)`, we look up `"read_file"`
   in the registry to find the actual Python function
2. **Documentation** -- we auto-generate tool descriptions from the functions'
   docstrings and type signatures, so the LLM knows what's available

This pattern scales beautifully: adding a new tool means writing a function
and adding one line to the registry.


In [37]:
# --- Tool Registry ---

TOOL_REGISTRY = {
    "read_file": read_file_tool,
    "list_files": list_files_tool,
    "edit_file": edit_file_tool,
    "run_bash": run_bash_tool,
}

def get_tool_description(tool_name: str) -> str:
    """Generate a human/LLM-readable description of a tool."""
    tool = TOOL_REGISTRY[tool_name]
    sig = inspect.signature(tool)
    doc = tool.__doc__
    return f"TOOL\n====\nName: {tool_name}\nDescription: {doc}\nSignature: {sig}\n==============="

# Let us see what the LLM will learn about our tools:
for name in TOOL_REGISTRY:
    print(get_tool_description(name))
    print()

TOOL
====
Name: read_file
Description: 
    Read the full content of a file.
    :param filename: The name or path of the file to read.
    :return: A dict with file_path and content keys.
    
Signature: (filename: str) -> Dict[str, Any]

TOOL
====
Name: list_files
Description: 
    List the files and directories at the given path.
    :param path: The path to a directory to list.
    :return: A dict with path and files keys. Each file entry
             has filename and type (file or dir).
    
Signature: (path: str) -> Dict[str, Any]

TOOL
====
Name: edit_file
Description: 
    Edit a file by replacing old_str with new_str. If old_str is empty,
    create (or overwrite) the file with new_str as its full content.
    :param path: The path to the file to edit or create.
    :param old_str: The exact string to find and replace (empty = create file).
    :param new_str: The replacement string (or full file content if creating).
    :return: A dict with path and action keys.
    
Signatu

## 6. The System Prompt

The system prompt is the "instruction manual" we give the LLM. It tells the model:
1. **Who it is** -- a coding assistant
2. **What tools it has** -- auto-generated from our registry
3. **How to call tools** -- the exact text format to use

The format we use is simple text-based tool calling:

```
tool: read_file({"filename": "main.py"})
```

> **Why text-based?** Many APIs now support native
> [function calling](https://platform.openai.com/docs/guides/function-calling),
> but the text-based approach is more educational -- you can see *exactly* what
> the LLM outputs and how parsing works.


In [38]:
SYSTEM_PROMPT_TEMPLATE = (
    "You are a coding assistant whose goal is to help solve coding tasks.\n"
    "You have access to a series of tools. Here are the tools you can use:\n\n"
    "{tool_list_repr}\n\n"
    "INSTRUCTIONS:\n"
    "- When you want to use a tool, reply with EXACTLY one line in this format:\n"
    "  tool: TOOL_NAME({{\"key\": \"value\"}})\n"
    "- Use compact single-line JSON with double quotes.\n"
    "- After receiving a tool_result(...) message, analyze the result and continue.\n"
    "- You may chain multiple tool calls across turns (one tool per turn).\n"
    "- If no tool is needed, respond normally with your answer.\n"
    "- Think step by step about what you need to do before acting.\n"
)

def get_full_system_prompt() -> str:
    """Build the complete system prompt with tool descriptions."""
    tool_descriptions = "\n\n".join(
        get_tool_description(name) for name in TOOL_REGISTRY
    )
    return SYSTEM_PROMPT_TEMPLATE.format(tool_list_repr=tool_descriptions)

# Let us see the full prompt the LLM will receive:
print(get_full_system_prompt())

You are a coding assistant whose goal is to help solve coding tasks.
You have access to a series of tools. Here are the tools you can use:

TOOL
====
Name: read_file
Description: 
    Read the full content of a file.
    :param filename: The name or path of the file to read.
    :return: A dict with file_path and content keys.
    
Signature: (filename: str) -> Dict[str, Any]

TOOL
====
Name: list_files
Description: 
    List the files and directories at the given path.
    :param path: The path to a directory to list.
    :return: A dict with path and files keys. Each file entry
             has filename and type (file or dir).
    
Signature: (path: str) -> Dict[str, Any]

TOOL
====
Name: edit_file
Description: 
    Edit a file by replacing old_str with new_str. If old_str is empty,
    create (or overwrite) the file with new_str as its full content.
    :param path: The path to the file to edit or create.
    :param old_str: The exact string to find and replace (empty = create file)

## 7. Parsing Tool Calls

When the LLM wants to use a tool, it outputs a line like:

```
tool: read_file({"filename": "main.py"})
```

We need to parse this to extract:
1. The **tool name** (`read_file`)
2. The **arguments** (`{"filename": "main.py"}`)

### Exercise 5: `extract_tool_invocations`

Implement the parser that extracts tool calls from LLM output text.

**Requirements:**
- Look for lines that start with `"tool:"` (case insensitive)
- Split on the first `(` to separate the tool name from the arguments
- The arguments are JSON between the first `(` and the last `)` on that line
- If the line doesn't end with `)`, skip it
- If the JSON is malformed, skip that line (no crash)
- Return a list of `(tool_name, args_dict)` tuples


In [45]:
def extract_tool_invocations(text: str) -> List[Tuple[str, Dict[str, Any]]]:
    """
    Parse tool invocations from LLM output.

    Looks for lines in the format:
        tool: tool_name({"arg1": "value1", "arg2": "value2"})

    Returns a list of (tool_name, args_dict) tuples.
    """
    invocations = []
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith("tool:"):
            try:
                tool_str = line[len("tool:"):].strip()
                open_paren_idx = tool_str.index("(")
                tool_name = tool_str[:open_paren_idx].strip()


                if not tool_str.endswith(")"):
                    continue

                json_str = tool_str[open_paren_idx + 1:-1].strip()
                args_dict = json.loads(json_str)
                invocations.append((tool_name, args_dict))
            except (ValueError, json.JSONDecodeError):

                continue
    return invocations

**Tests for Exercise 5:** Run this cell to check your `extract_tool_invocations` implementation.

In [46]:
print("Testing extract_tool_invocations...")

def test_basic_tool_call():
    result = extract_tool_invocations('tool: read_file({"filename": "main.py"})')
    assert len(result) == 1, f"Expected 1 invocation, got {len(result)}"
    name, args = result[0]
    assert name == "read_file", f"Expected 'read_file', got {name!r}"
    assert args == {"filename": "main.py"}, f"Wrong args: {args}"

def test_multi_arg_tool_call():
    text = 'tool: edit_file({"path": "f.py", "old_str": "old", "new_str": "new"})'
    result = extract_tool_invocations(text)
    assert len(result) == 1
    name, args = result[0]
    assert name == "edit_file"
    assert args["path"] == "f.py"
    assert args["old_str"] == "old"
    assert args["new_str"] == "new"

def test_no_tool_call():
    result = extract_tool_invocations("This is a normal response, no tool call.")
    assert result == [], f"Expected empty list, got {result}"

def test_mixed_text_and_tool():
    text = "Let me read that file for you.\ntool: read_file({\"filename\": \"test.py\"})"
    result = extract_tool_invocations(text)
    assert len(result) == 1
    assert result[0][0] == "read_file"

def test_malformed_json_skipped():
    result = extract_tool_invocations("tool: read_file({bad json})")
    assert result == [], f"Malformed JSON should be skipped, got {result}"

def test_missing_closing_paren():
    result = extract_tool_invocations('tool: read_file({"filename": "main.py"}')
    assert result == [], f"Missing closing paren should be skipped, got {result}"

def test_case_insensitive():
    result = extract_tool_invocations('Tool: read_file({"filename": "main.py"})')
    assert len(result) == 1, f"Should handle 'Tool:' (capitalized), got {result}"
    assert result[0][0] == "read_file"

def test_returns_list_of_tuples():
    result = extract_tool_invocations('tool: list_files({"path": "."})')
    assert isinstance(result, list), f"Expected list, got {type(result)}"
    assert len(result) == 1
    assert isinstance(result[0], tuple), f"Expected tuple, got {type(result[0])}"
    assert len(result[0]) == 2, f"Expected 2-tuple, got {len(result[0])}-tuple"

run_test("basic tool call", test_basic_tool_call)
run_test("multi-arg tool call", test_multi_arg_tool_call)
run_test("no tool call in text", test_no_tool_call)
run_test("mixed text and tool call", test_mixed_text_and_tool)
run_test("malformed JSON skipped", test_malformed_json_skipped)
run_test("missing closing paren skipped", test_missing_closing_paren)
run_test("case insensitive (Tool:)", test_case_insensitive)
run_test("returns list of (name, args) tuples", test_returns_list_of_tuples)
show_results()

Testing extract_tool_invocations...
  PASS: basic tool call
  PASS: multi-arg tool call
  PASS: no tool call in text
  PASS: mixed text and tool call
  PASS: malformed JSON skipped
  PASS: missing closing paren skipped
  PASS: case insensitive (Tool:)
  PASS: returns list of (name, args) tuples

  8/8 tests passed
  All tests passed!


## 8. The LLM Call

This is where we connect to the actual LLM via OpenRouter. This function is
provided for you -- it simply passes the conversation history and gets a response.

> **Model choice matters:** Different models have different strengths.
> For coding tasks, [Qwen3-Coder](https://qwenlm.github.io/blog/qwen3/) (480B MoE, free)
> is excellent. For reasoning, [DeepSeek R1](https://deepseek.com/) is strong.


In [47]:
def execute_llm_call(conversation: List[Dict[str, str]]) -> str:
    """
    Send the conversation to the LLM and return its response text.

    The conversation is a list of messages, each with 'role' and 'content'.
    Roles: 'system', 'user', 'assistant'
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=conversation,
        max_tokens=2000,
        temperature=0.2,  # Low temperature for more reliable tool calls
    )
    # Some models may return None content; default to empty string
    return response.choices[0].message.content or ""

# Quick test
test_conv = [
    {"role": "system", "content": "You are a helpful assistant. Be very brief."},
    {"role": "user", "content": "What is 2 + 2?"}
]
print(f"LLM says: {execute_llm_call(test_conv)}")

LLM says: 2 + 2 = 4


## 9. The Agent Loop -- Where the Magic Happens

The loop works like this:
1. Take a user task (a string describing what to do)
2. Send it to the LLM along with the system prompt
3. Check the LLM's response for tool calls
4. If tools are requested: execute them, add results to conversation, **go back to step 3**
5. If no tools: the agent is done, return the final response

Here's what [David Crawshaw](https://crawshaw.io/blog/programming-with-agents)
(co-founder of Tailscale) says about this pattern:

> *"An agent is a for loop which contains an LLM call, where the LLM can
> execute commands and see their output without a human in the loop."*

### Tool Dispatch (provided)

The tool dispatch map and executor are provided. These translate tool names and
arguments from the parsed output into actual function calls.


In [48]:
# Map tool names to their functions and expected argument names.
# This lets us dispatch dynamically -- adding a new tool means adding one entry here.
TOOL_ARG_MAP = {
    "read_file":  lambda a: read_file_tool(a["filename"]),
    "list_files": lambda a: list_files_tool(a["path"]),
    "edit_file":  lambda a: edit_file_tool(a["path"], a.get("old_str", ""), a.get("new_str", "")),
    "run_bash":   lambda a: run_bash_tool(a["command"], a.get("timeout", 30)),
}

def execute_tool(name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    """Execute a tool by name. Returns an error dict if anything goes wrong."""
    if name not in TOOL_ARG_MAP:
        return {"error": f"Unknown tool: {name}"}
    try:
        return TOOL_ARG_MAP[name](args)
    except Exception as e:
        # Return the error to the LLM so it can recover gracefully
        return {"error": f"{type(e).__name__}: {e}"}

print("Tool dispatch ready!")

Tool dispatch ready!


### Exercise 6: `run_agent` -- The Core Agent Loop

This is the heart of the whole system! Implement the agentic loop.

**The conversation is initialized for you.** Your job is to implement the loop body.

**Requirements:**
- Loop up to `max_turns` times
- Each turn: call `execute_llm_call(conversation)`, append the response as an assistant message
- Parse tool calls with `extract_tool_invocations()`
- If no tool calls: return the assistant's response (the agent is done)
- If there are tool calls: execute each with `execute_tool()`, append results as user
  messages in the format `f"tool_result({json.dumps(result)})"`
- If the loop exhausts `max_turns`, return `"Agent hit maximum turn limit."`


In [53]:
def run_agent(task: str, max_turns: int = 15, verbose: bool = True) -> str:
    """
    Run the coding agent on a task.

    Args:
        task:      Natural language description of the coding task.
        max_turns: Maximum number of LLM calls (safety limit).
        verbose:   Whether to print step-by-step progress.

    Returns:
        The agent's final text response.
    """
    # Initialize conversation with system prompt + user task
    conversation = [
        {"role": "system", "content": get_full_system_prompt()},
        {"role": "user", "content": task},
    ]

    if verbose:
        print("=" * 60)
        print(f"  TASK: {task[:80]}")
        print("=" * 60)

    # TODO: Implement the agentic loop
    # Hints:
    #   1. for turn in range(max_turns):
    #   2.   Call execute_llm_call(conversation) to get the assistant's response
    #   3.   Append {"role": "assistant", "content": response} to conversation
    #   4.   Parse tool calls: tool_invocations = extract_tool_invocations(response)
    #   5.   If no tool_invocations: return the response (agent is done)
    #   6.   For each (name, args) in tool_invocations:
    #          - If verbose, print the tool call
    #          - result = execute_tool(name, args)
    #          - Append {"role": "user", "content": f"tool_result({json.dumps(result)})"} to conversation
    #   7. After the loop, return "Agent hit maximum turn limit."
    for turn in range(max_turns):
        response = execute_llm_call(conversation)
        conversation.append({"role": "assistant", "content": response})

        tool_invocations = extract_tool_invocations(response)

        if not tool_invocations:
            return response

        for name, args in tool_invocations:
            if verbose:
                print(f"  [tool] {name}({json.dumps(args)}) ")
            result = execute_tool(name, args)
            if verbose:
                result_str = json.dumps(result)
                if len(result_str) > 200:
                    result_str = result_str[:197] + "..."
                print(f"     -> {result_str}")
            conversation.append({"role": "user", "content": f"tool_result({json.dumps(result)})"})

    return "Agent hit maximum turn limit."



**Tests for Exercise 6:** These tests use a **simulated LLM** so they work without
an API key. We monkey-patch `execute_llm_call` with scripted responses to verify
the loop logic.


In [54]:
print("Testing run_agent...")

# Save the real execute_llm_call so we can restore it
_real_execute_llm_call = execute_llm_call

def _make_fake_llm(scripted_responses):
    """Create a fake LLM that returns scripted responses in order."""
    responses = list(scripted_responses)
    def fake_llm(conversation):
        if responses:
            return responses.pop(0)
        return "No more scripted responses."
    return fake_llm

def test_agent_creates_file():
    """Test that the agent loop actually executes tools."""
    global execute_llm_call
    # Script: LLM asks to create a file, then responds with done
    execute_llm_call = _make_fake_llm([
        'I will create the file.\ntool: edit_file({"path": "_agent_test.txt", "old_str": "", "new_str": "hello from agent"})',
        "Done! I created the file.",
    ])
    try:
        result = run_agent("Create _agent_test.txt", max_turns=5, verbose=False)
        # The file should actually exist
        test_path = WORKSPACE / "_agent_test.txt"
        assert test_path.exists(), "Agent should have created the file"
        assert test_path.read_text() == "hello from agent", \
            f"File content wrong: {test_path.read_text()!r}"
        test_path.unlink()
    finally:
        execute_llm_call = _real_execute_llm_call

def test_agent_returns_final_response():
    """Test that the agent returns the LLM's text when no tool call is made."""
    global execute_llm_call
    execute_llm_call = _make_fake_llm([
        "Here is your answer: 42",
    ])
    try:
        result = run_agent("What is the answer?", max_turns=5, verbose=False)
        assert "42" in result, f"Expected '42' in result: {result!r}"
    finally:
        execute_llm_call = _real_execute_llm_call

def test_agent_respects_max_turns():
    """Test that the agent stops after max_turns even if LLM keeps calling tools."""
    global execute_llm_call
    # Every response is a tool call -- the agent should stop after max_turns
    execute_llm_call = _make_fake_llm([
        'tool: list_files({"path": "."})',
        'tool: list_files({"path": "."})',
        'tool: list_files({"path": "."})',
        'tool: list_files({"path": "."})',
        'tool: list_files({"path": "."})',
    ])
    try:
        result = run_agent("Loop forever", max_turns=3, verbose=False)
        assert "maximum turn limit" in result.lower(), \
            f"Expected turn limit message, got: {result!r}"
    finally:
        execute_llm_call = _real_execute_llm_call

run_test("agent executes tools (creates file)", test_agent_creates_file)
run_test("agent returns final text response", test_agent_returns_final_response)
run_test("agent respects max_turns limit", test_agent_respects_max_turns)
show_results()

Testing run_agent...
  PASS: agent executes tools (creates file)
  PASS: agent returns final text response
  PASS: agent respects max_turns limit

  3/3 tests passed
  All tests passed!


## 10. Let's Run It!

**Congratulations!** If all your tests pass, you have a fully working coding agent!

Time to see it in action. We'll give it real coding tasks and watch it work step by step.

### Demo 1: Create a Python Project

Let's ask the agent to create a small Python project from scratch.

> **Try it:** After this cell runs, open the **Files** panel on the left
> and navigate into `agent_workspace/` -- you should see the files the agent created!


In [55]:
# Clean the workspace for our demo
import shutil
for item in WORKSPACE.iterdir():
    if item.is_file():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

# --- DEMO 1: Create a project and run it ---
run_agent(
    "Create a Python file called 'calculator.py' with a Calculator class "
    "that has methods for add, subtract, multiply, and divide (handle "
    "division by zero). Then create a 'test_calculator.py' that tests "
    "all the methods using assert statements. "
    "Run the tests with 'python test_calculator.py' to verify they pass. "
    "Finally, list the files to confirm they were created."
)

  TASK: Create a Python file called 'calculator.py' with a Calculator class that has met
  [tool] edit_file({"path": "calculator.py", "old_str": "", "new_str": "class Calculator:\n    \"\"\"A simple calculator class with basic arithmetic operations.\"\"\"\n    \n    def add(self, a, b):\n        \"\"\"Add two numbers and return the result.\"\"\"\n        return a + b\n    \n    def subtract(self, a, b):\n        \"\"\"Subtract b from a and return the result.\"\"\"\n        return a - b\n    \n    def multiply(self, a, b):\n        \"\"\"Multiply two numbers and return the result.\"\"\"\n        return a * b\n    \n    def divide(self, a, b):\n        \"\"\"Divide a by b and return the result.\n        \n        Raises:\n            ValueError: If b is zero (division by zero).\n        \"\"\"\n        if b == 0:\n            raise ValueError(\"Cannot divide by zero\")\n        return a / b\n"}) 
     -> {"path": "/content/agent_workspace/calculator.py", "action": "created_file"}


'Now let me create the test_calculator.py file:\n\n```python\ntool: edit_file({"path": "test_calculator.py", "old_str": "", "new_str": "import pytest\\nfrom calculator import Calculator\\n\\n\\nclass TestCalculator:\\n    \\"\\"\\"Test class for the Calculator class.\\"\\"\\"\\n    \\n    def setup_method(self):\\n        \\"\\"\\"Set up test fixtures.\\"\\"\\"\\n        self.calc = Calculator()\\n    \\n    def test_add(self):\\n        \\"\\"\\"Test the add method.\\"\\"\\"\\n        assert self.calc.add(2, 3) == 5\\n        assert self.calc.add(-1, 1) == 0\\n        assert self.calc.add(0, 0) == 0\\n        assert self.calc.add(1.5, 2.5) == 4.0\\n    \\n    def test_subtract(self):\\n        \\"\\"\\"Test the subtract method.\\"\\"\\"\\n        assert self.calc.subtract(5, 3) == 2\\n        assert self.calc.subtract(1, 1) == 0\\n        assert self.calc.subtract(-1, -1) == 0\\n        assert self.calc.subtract(0, 5) == -5\\n    \\n    def test_multiply(self):\\n        \\"\\"\\"Test

In [ ]:
# Let's see what the agent created:
for name in sorted(WORKSPACE.iterdir()):
    if name.is_file():
        print(f"{'=' * 50}")
        print(f"  {name.name}")
        print(f"{'=' * 50}")
        print(name.read_text())

### Demo 2: Read and Refactor

Now let's ask the agent to read the code it created and improve it.


In [ ]:
# --- DEMO 2: Read and refactor ---
run_agent(
    "Read calculator.py, then add a 'power' method that raises a number "
    "to a given exponent, and a 'sqrt' method that returns the square root. "
    "Import math if needed."
)

# Show the updated file
print(WORKSPACE.joinpath("calculator.py").read_text())

### Demo 3: Explore and Fix

Let's ask the agent to find and fix an issue.


In [ ]:
# --- DEMO 3: Explore and fix ---
run_agent(
    "List all files in the workspace, read test_calculator.py, and add "
    "tests for the new power and sqrt methods that were just added to "
    "calculator.py. Read calculator.py first to see the current methods."
)

# Show the updated test file
print(WORKSPACE.joinpath("test_calculator.py").read_text())

## 11. Interactive Mode (Optional)

Want to chat with your agent interactively? Run the cell below.
Type your messages and press Enter. Type `quit` or `exit` to stop.

> **Note:** This uses `input()` which works in Colab but blocks the cell.
> The non-interactive `run_agent()` function above is usually better for notebooks.


In [ ]:
def run_interactive():
    """Run the agent in interactive chat mode."""
    conversation = [{"role": "system", "content": get_full_system_prompt()}]

    print("Coding Agent (interactive mode)")
    print("    Type your request, or 'quit' to exit.\n")

    while True:
        try:
            user_input = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            break
        if user_input.lower() in ("quit", "exit", "q"):
            break
        if not user_input:
            continue

        conversation.append({"role": "user", "content": user_input})

        while True:
            response = execute_llm_call(conversation)
            conversation.append({"role": "assistant", "content": response})
            tools = extract_tool_invocations(response)

            if not tools:
                print(f"\nAssistant: {response}\n")
                break

            for name, args in tools:
                print(f"  [tool] {name}({json.dumps(args)})")
                result = execute_tool(name, args)
                print(f"     -> {json.dumps(result)[:200]}")
                conversation.append({
                    "role": "user",
                    "content": f"tool_result({json.dumps(result)})"
                })

    print("\nSession ended.")

# Uncomment the line below to start interactive mode:
# run_interactive()

## 12. From Toy to Production -- What Real Agents Add

Our agent works! But production tools like Claude Code, Cursor, and Aider
add several layers on top of this core loop:

| Feature | What it does | Why it matters |
|---------|-------------|----------------|
| **Streaming** | Show tokens as they arrive | Better UX, feels responsive |
| **Context management** | Summarize/compress old messages | LLMs have finite context windows |
| **More tools** | grep, bash, web search, git | More capabilities = more useful |
| **Sandboxing** | Run code in containers | Safety -- don't let the agent `rm -rf /` |
| **Error recovery** | Retry failed tool calls | Robustness in the real world |
| **Human-in-the-loop** | Ask for confirmation on risky ops | Safety for destructive actions |
| **Codebase indexing** | Embed/index the whole repo | Find relevant code faster |
| **Native tool calling** | Use API-level function calling | More reliable than text parsing |

But the **core pattern is identical** to what we built. As
[Thorsten Ball (Amp/Sourcegraph)](https://ampcode.com/notes/how-to-build-an-agent) wrote:

> *"A fully functioning code-editing agent can be built in under 400 lines of code,
> most of which is boilerplate."*

And Princeton's [mini-swe-agent](https://github.com/SWE-agent/mini-swe-agent) -- just 100 lines
of Python -- scores over 74% on [SWE-bench Verified](https://www.swebench.com/),
a benchmark of real GitHub issues from popular Python projects. To put that in perspective:
the most sophisticated commercial agents (with thousands of lines of scaffolding, RAG
pipelines, and custom tooling) top out around 75-80%. A 100-line script gets
within a few percentage points of the state of the art.

**The lesson:** the intelligence comes from the LLM, not from complex scaffolding.
Your job as the engineer is to give the LLM good tools, clear instructions, and
a reliable loop.


## 13. References and Further Reading

### The "Emperor Has No Clothes" Thesis
- **[The Emperor Has No Clothes](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)** -- Mihail Eric.
  *The article this tutorial is based on. Shows how to build the core of Claude Code in ~200 lines.*
- **[How to Build an Agent](https://ampcode.com/notes/how-to-build-an-agent)** -- Thorsten Ball (Amp/Sourcegraph).
  *The blog post that crystallized the simplicity thesis.*
- **[Amp: The Emperor Has No Clothes](https://www.latent.space/p/amp-the-emperor-has-no-clothes)** -- Latent Space Podcast.
  *Deep dive into why coding agents are simpler than you think.*
- **[Coding Agent in 94 Lines of Ruby](https://radanskoric.com/articles/coding-agent-in-ruby)** -- Radan Skoric.
  *Proves the pattern is language-agnostic.*

### Foundational Papers
- **[ReAct: Synergizing Reasoning and Acting](https://arxiv.org/abs/2210.03629)** -- Yao et al. (ICLR 2023).
  *The foundational paper for the observe-think-act loop used in agents.*
- **[Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)** -- Wei et al. (NeurIPS 2022).
  *Why "think step by step" works -- the intellectual predecessor to ReAct.*
- **[Toolformer](https://arxiv.org/abs/2302.04761)** -- Schick et al. (NeurIPS 2023).
  *LLMs can learn when and how to call external tools.*

### Agent Design Guides
- **[Building Effective Agents](https://www.anthropic.com/research/building-effective-agents)** -- Anthropic.
  *The single best industry guide to agent architecture. "Find the simplest solution possible."*
- **[Effective Context Engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)** -- Anthropic.
  *How to manage the information an agent sees -- the core engineering challenge.*
- **[Writing Tools for Agents](https://www.anthropic.com/engineering/writing-tools-for-agents)** -- Anthropic.
  *Tool descriptions matter more than you think. Practical guide from the Claude Code team.*
- **[How I Program with Agents](https://crawshaw.io/blog/programming-with-agents)** -- David Crawshaw.
  *"An agent is a for loop which contains an LLM call."*

### Open Source Coding Agents
- **[pi](https://github.com/badlogic/pi-mono/tree/main/packages/coding-agent)** -- Mario Zechner's aggressively extensible coding agent. Just 4 default tools (read, write, edit, bash). Great example of minimal-core-with-extensions design.
- **[Aider](https://github.com/Aider-AI/aider)** -- AI pair programming in the terminal. Mature, multi-LLM, git-aware.
- **[OpenHands](https://github.com/OpenHands/OpenHands)** (formerly OpenDevin) -- Full sandboxed agent platform. Top SWE-bench scores.
- **[SWE-agent](https://github.com/SWE-agent/SWE-agent)** -- Princeton's agent that solves real GitHub issues.
- **[mini-swe-agent](https://github.com/SWE-agent/mini-swe-agent)** -- 100 lines of Python, 74%+ on SWE-bench Verified (commercial agents with 1000s of lines score ~75-80%). The ultimate simplicity proof.

### Tool Calling and APIs
- **[OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling)** -- The canonical reference.
- **[Anthropic Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview)** -- Claude-specific guide.
- **[Martin Fowler on Function Calling](https://martinfowler.com/articles/function-call-LLM.html)** -- Architecture-focused explanation.
- **[Model Context Protocol (MCP)](https://modelcontextprotocol.io/)** -- Emerging open standard for tool integration.

### Learning Resources
- **[Hugging Face Agents Course](https://huggingface.co/learn/agents-course/unit1/agent-steps-and-structure)** -- Free hands-on course.
- **[DeepLearning.AI smolagents Course](https://learn.deeplearning.ai/courses/building-code-agents-with-hugging-face-smolagents)** -- Short free video course.
- **[Prompt Engineering Guide](https://www.promptingguide.ai/)** -- Comprehensive guide covering ReAct, CoT, and tool use.
- **[IBM: What is a ReAct Agent?](https://www.ibm.com/think/topics/react-agent)** -- Gentle, accessible introduction.
- **[OpenRouter Free Models](https://openrouter.ai/models?fmt=cards&input_modalities=text&max_price=0&order=most-popular)** -- Browse all available free models.


---

## Summary

Congratulations! You built a coding agent from scratch by implementing 6 key functions:

1. **`read_file_tool`** -- Let the agent see code
2. **`list_files_tool`** -- Let the agent navigate
3. **`edit_file_tool`** -- Let the agent write code (create + edit modes)
4. **`run_bash_tool`** -- Let the agent execute commands
5. **`extract_tool_invocations`** -- Parse LLM output into tool calls
6. **`run_agent`** -- The core agentic loop tying it all together

Here is what you learned:

1. **The agentic loop** is a `for` loop with an LLM call and tool execution
2. **Four tools** (read, list, edit, bash) give you a surprisingly capable agent
3. **Tool descriptions** (docstrings + signatures) teach the LLM what is available
4. **Text-based tool calling** is simple to parse and works with any model
5. **The intelligence comes from the model** -- the scaffolding is simple

The gap between this tutorial and production tools like Claude Code, Cursor, or Aider
is mainly about **polish** (streaming, error handling, sandboxing, context management) --
not about fundamental architecture. The emperor truly has no clothes.

---

*Built with [OpenRouter](https://openrouter.ai) free models.
Based on [The Emperor Has No Clothes](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)
by [Mihail Eric](https://www.mihaileric.com/).*
